# GEO 371T/391: Climate Data - Spring 2026
# Assignment 8
# Model Comparison of Cumulative Precipitation Metrics

*Developed by Cameron Cummins: 3/6/2026*

*Updated by Geeta Persad: 3/9/2026*

**In this notebook, we will focus on the data analysis needed to answer the core question of the model comparison step of our class project: What is the historical distribution of cumulative precipitation from models and how well do they agree with observations?**

**To do this, we will map, over our 30-year historical baseline period (1985-2014) for 1-day, 3-day, and 5-day cumulative precipitation:**
1. The 95th percentile precipitation amount
2. The 30-year trend

**On each map, you will notice there is stippling (dots). Locations where there is a dot are locations where the model value falls within the range of values from our four observational datasets.**

**You will have the following tasks in this assignment, some of which we will work on in lecture and some of which you will complete outside of class time:**

- Task 1: Given 95th percentile map, add a figure caption
- Task 2: Given trend map, add a figure caption
- Task 3: Save out your notebook as an html file and separately save out your figures by right clicking on them. Then complete the Assignment 8 template on Canvas, which prompts you to:
    - Write a documentation paragraph for your assigned climate model following the provided prompts.
    - Choose a region of interest and, based on the stippling on the two figures, select and justify a "best" model for simulating the 95th percentile and trend in cumulative precipitation for that region.


## Step 1: Run the below two code blocks

### The first code block loads in our packages and data and defines our mapping projection.

In [ ]:
%%time
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib
from os import listdir
matplotlib.style.use('fast')


class WinkelTripel(ccrs._WarpedRectangularProjection):
	"""
	Winkel-Tripel projection implementation for Cartopy
	"""

	def __init__(self, central_longitude=0.0, central_latitude=0.0, globe=None):
		globe = globe or ccrs.Globe(semimajor_axis=ccrs.WGS84_SEMIMAJOR_AXIS)
		proj4_params = [('proj', 'wintri'),
						('lon_0', central_longitude),
						('lat_0', central_latitude)]

		super(WinkelTripel, self).__init__(proj4_params, central_longitude, globe=globe)

	@property
	def threshold(self):
		return 1e4

head_input_dir = "/scratch/07644/oxygen/GEO371T_Post_Processed/cmip_statistics"
quant_paths = [f"{head_input_dir}/{name}" for name in listdir(head_input_dir) if "_quants.nc" in name and "hist" in name]
trends_paths = [f"{head_input_dir}/{name}" for name in listdir(head_input_dir) if "_trends.nc" in name and "hist" in name]

models = [name.split("/")[-1].split("_")[0] for name in quant_paths]
quants_ds = xr.open_mfdataset(quant_paths, combine="nested", concat_dim="model").assign_coords(dict(model=models))*86400

models = [name.split("/")[-1].split("_")[0] for name in trends_paths]
trends_ds = xr.open_mfdataset(trends_paths, combine="nested", concat_dim="model").assign_coords(dict(model=models))*86400

### This second code block loads in the observational datasets that we used last week to allow us to check how the models compare with the observations.

In [ ]:
obs_input_dir = "/scratch/07644/oxygen/GEO371T_Post_Processed/obs_statistics"

era5_ds = xr.open_dataset(f"{obs_input_dir}/ERA5_quantiles.nc").interp(lat=quants_ds.lat, lon=quants_ds.lon)
mswep_g_ds = xr.open_dataset(f"{obs_input_dir}/MSWEP_G_quantiles.nc").interp(lat=quants_ds.lat, lon=quants_ds.lon)
mswep_ng_ds = xr.open_dataset(f"{obs_input_dir}/MSWEP_NG_quantiles.nc").interp(lat=quants_ds.lat, lon=quants_ds.lon)
noaa_cpc_ds = xr.open_dataset(f"{obs_input_dir}/NOAA_CPC_quantiles.nc").interp(lat=quants_ds.lat, lon=quants_ds.lon)
obs_quants = xr.concat([era5_ds, mswep_g_ds, mswep_ng_ds, noaa_cpc_ds], dim="model")
obs_quants_min = obs_quants.min(dim="model")
obs_quants_max = obs_quants.max(dim="model")

era5_ds = xr.open_dataset(f"{obs_input_dir}/ERA5_trends.nc").interp(lat=trends_ds.lat, lon=trends_ds.lon)
mswep_g_ds = xr.open_dataset(f"{obs_input_dir}/MSWEP_G_trends.nc").interp(lat=trends_ds.lat, lon=trends_ds.lon)
mswep_ng_ds = xr.open_dataset(f"{obs_input_dir}/MSWEP_NG_trends.nc").interp(lat=trends_ds.lat, lon=trends_ds.lon)
noaa_cpc_ds = xr.open_dataset(f"{obs_input_dir}/NOAA_CPC_trends.nc").interp(lat=trends_ds.lat, lon=trends_ds.lon)
obs_trends = xr.concat([era5_ds, mswep_g_ds, mswep_ng_ds, noaa_cpc_ds], dim="model")
obs_trends_min = obs_trends.min(dim="model")
obs_trends_max = obs_trends.max(dim="model")

# Step 2: Map the 95th percentile

The code below contains all the steps needed to map the 95th percentile values of each metric over the 30-year period in each of the 9 models. 

Run the below block of code to generate the figure.

## Task 1: Add a Figure Caption

Below the 95th percentile figure is a Markdown cell with the start of a figure caption. Use the best practices discussed in class to write a complete figure caption. Make sure to define the stippling in your caption.

In [ ]:
%matplotlib inline
####################################
##         Quantiles Map          ##
####################################

def get_stipple_mask(model_da, lower_da, upper_da):
    return (model_da >= lower_da) & (model_da <= upper_da)

proj = WinkelTripel()
transform = ccrs.PlateCarree()
cmap = "BuPu"
quantile = 0.95
robust = True

one_day_levels = None# np.arange(0, 41, 5)
three_day_levels = None# np.arange(0, 41, 5)
five_day_levels = None# np.arange(0, 41, 5)
fz = 26
pad = 20
stipple_size = 1
stipple_spacing = 2

cb_labels = [
    "mm/day", "mm/3day", "mm/5day"
]

f, axes = plt.subplots(9, 3, figsize=(30, 40), facecolor='w', subplot_kw=dict(projection=proj))

mask = get_stipple_mask(quants_ds, obs_quants_min, obs_quants_max)
lons = mask.lon.values
lats = mask.lat.values
lon_grid, lat_grid = np.meshgrid(lons, lats)
for index, model in enumerate(quants_ds.model.values):
    quants_ds["one_day_pr"].rename(cb_labels[0]).sel(model=model, quantile=quantile).plot.imshow(ax=axes[index,0], levels=one_day_levels, transform=transform, cmap=cmap, robust=robust)
    quants_ds["three_day_pr"].rename(cb_labels[1]).sel(model=model, quantile=quantile).plot.imshow(ax=axes[index,1], levels=three_day_levels, transform=transform, cmap=cmap, robust=robust)
    quants_ds["five_day_pr"].rename(cb_labels[2]).sel(model=model, quantile=quantile).plot.imshow(ax=axes[index,2], levels=five_day_levels, transform=transform, cmap=cmap, robust=robust)

    model_mask = mask.sel(model=model, quantile=quantile)
    
    stipple_lons = lon_grid[::stipple_spacing, ::stipple_spacing][model_mask["one_day_pr"].values[::stipple_spacing, ::stipple_spacing]]
    stipple_lats = lat_grid[::stipple_spacing, ::stipple_spacing][model_mask["one_day_pr"].values[::stipple_spacing, ::stipple_spacing]]
    axes[index, 0].scatter(stipple_lons, stipple_lats, transform=transform, s=stipple_size, color="black", alpha=1, marker=".")

    stipple_lons = lon_grid[::stipple_spacing, ::stipple_spacing][model_mask["three_day_pr"].values[::stipple_spacing, ::stipple_spacing]]
    stipple_lats = lat_grid[::stipple_spacing, ::stipple_spacing][model_mask["three_day_pr"].values[::stipple_spacing, ::stipple_spacing]]
    axes[index, 1].scatter(stipple_lons, stipple_lats, transform=transform, s=stipple_size, color="black", alpha=1, marker=".")

    stipple_lons = lon_grid[::stipple_spacing, ::stipple_spacing][model_mask["five_day_pr"].values[::stipple_spacing, ::stipple_spacing]]
    stipple_lats = lat_grid[::stipple_spacing, ::stipple_spacing][model_mask["five_day_pr"].values[::stipple_spacing, ::stipple_spacing]]
    axes[index, 2].scatter(stipple_lons, stipple_lats, transform=transform, s=stipple_size, color="black", alpha=1, marker=".")
    
    axes[index, 0].text(-0.2, 0.1 + (-1.2*index), model, rotation=90, transform=axes[0, 0].transAxes, fontsize=22)

for i in range(axes.shape[0]):
    for j in range(axes.shape[1]):
        axes[i, j].coastlines()
        axes[i, j].set_title("")

axes[0, 0].set_title("1-Day Precp.", fontsize=fz, pad=pad)
axes[0, 1].set_title("3-Day Precp.", fontsize=fz, pad=pad)
axes[0, 2].set_title("5-Day Precp.", fontsize=fz, pad=pad)

f.suptitle(f"{int(quantile*100)}th Percentile Precip. Metrics for CMIP6 Datasets", fontsize=45)
f.show()

### Figure 1: 

# Step 3: Map the Trends

The below block of code maps the 30-year trends in each of the 3 metrics for each of the 9 observational datasets.

The 30-year trends are calculated by performing a linear regression through the **annual maximum** value of each metric. The maps show the rate of change in each year's wettest multi-day event over the 30 year period.

## Task 2: Add a Figure Caption
Below the trend figure is a Markdown cell with the start of a figure caption. Use the best practices discussed in class to write a complete figure caption. Make sure to define the stippling in your caption.

In [ ]:
%matplotlib inline
####################################
##           Trends Map           ##
####################################

def get_stipple_mask(model_da, lower_da, upper_da):
    return (model_da >= lower_da) & (model_da <= upper_da)

proj = WinkelTripel()
transform = ccrs.PlateCarree()
cmap = "BuPu"

one_day_levels = None#np.arange(0, 301, 50)
three_day_levels = None#np.arange(0, 301, 50)
five_day_levels = None#np.arange(0, 301, 50)
robust = True
fz = 26
pad = 20
stipple_size = 1
stipple_spacing = 2

cb_labels = [
    "Δ(mm/day) yr-1", "Δ(mm/3day) yr-1", "Δ(mm/5day) yr-1"
]

f, axes = plt.subplots(9, 3, figsize=(30, 40), facecolor='w', subplot_kw=dict(projection=proj))

mask = get_stipple_mask(trends_ds, obs_trends_min, obs_trends_max)
lons = mask.lon.values
lats = mask.lat.values
lon_grid, lat_grid = np.meshgrid(lons, lats)
for index, model in enumerate(trends_ds.model.values):
    trends_ds["one_day_pr_polyfit_coefficients"].rename(cb_labels[0]).sel(model=model, degree=0).plot.imshow(ax=axes[index,0], levels=one_day_levels, transform=transform, cmap=cmap, robust=robust)
    trends_ds["three_day_pr_polyfit_coefficients"].rename(cb_labels[1]).sel(model=model, degree=0).plot.imshow(ax=axes[index,1], levels=three_day_levels, transform=transform, cmap=cmap, robust=robust)
    trends_ds["five_day_pr_polyfit_coefficients"].rename(cb_labels[2]).sel(model=model, degree=0).plot.imshow(ax=axes[index,2], levels=five_day_levels, transform=transform, cmap=cmap, robust=robust)
    
    model_mask = mask.sel(model=model, degree=0)
    
    stipple_lons = lon_grid[::stipple_spacing, ::stipple_spacing][model_mask["one_day_pr_polyfit_coefficients"].values[::stipple_spacing, ::stipple_spacing]]
    stipple_lats = lat_grid[::stipple_spacing, ::stipple_spacing][model_mask["one_day_pr_polyfit_coefficients"].values[::stipple_spacing, ::stipple_spacing]]
    axes[index, 0].scatter(stipple_lons, stipple_lats, transform=transform, s=stipple_size, color="black", alpha=1, marker=".")

    stipple_lons = lon_grid[::stipple_spacing, ::stipple_spacing][model_mask["three_day_pr_polyfit_coefficients"].values[::stipple_spacing, ::stipple_spacing]]
    stipple_lats = lat_grid[::stipple_spacing, ::stipple_spacing][model_mask["three_day_pr_polyfit_coefficients"].values[::stipple_spacing, ::stipple_spacing]]
    axes[index, 1].scatter(stipple_lons, stipple_lats, transform=transform, s=stipple_size, color="black", alpha=1, marker=".")

    stipple_lons = lon_grid[::stipple_spacing, ::stipple_spacing][model_mask["five_day_pr_polyfit_coefficients"].values[::stipple_spacing, ::stipple_spacing]]
    stipple_lats = lat_grid[::stipple_spacing, ::stipple_spacing][model_mask["five_day_pr_polyfit_coefficients"].values[::stipple_spacing, ::stipple_spacing]]
    axes[index, 2].scatter(stipple_lons, stipple_lats, transform=transform, s=stipple_size, color="black", alpha=1, marker=".")
    
    axes[index, 0].text(-0.2, 0.1 + (-1.2*index), model, rotation=90, transform=axes[0, 0].transAxes, fontsize=22)

    
for i in range(axes.shape[0]):
    for j in range(axes.shape[1]):
        axes[i, j].coastlines()
        axes[i, j].set_title("")

axes[0, 0].set_title("1-Day Precp.", fontsize=fz, pad=pad)
axes[0, 1].set_title("3-Day Precp.", fontsize=fz, pad=pad)
axes[0, 2].set_title("5-Day Precp.", fontsize=fz, pad=pad)

f.suptitle(f"Annual Trends of Precip. Metrics for CMIP6 Datasets", fontsize=45)
f.show()

### Figure 2: 

## Task 3: Save out your figures as images (right-click and save image as...) and save out your completed notebook as an html file. Then transition to the Assignment 8 Template on Canvas to complete the assignment.